In [1]:
!pip uninstall -y pillow
!pip install pillow==11.3.0
!pip install -U inference-sdk

Found existing installation: pillow 12.2.0
Uninstalling pillow-12.2.0:
  Successfully uninstalled pillow-12.2.0
  Using cached pillow-11.3.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.0 kB)
Using cached pillow-11.3.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (6.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inference-sdk 1.2.9 requires pillow<13.0.0,>=12.2.0, but you have pillow 11.3.0 which is incompatible.


  Using cached pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
Using cached pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (7.1 MB)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [2]:
!sudo apt install tesseract-ocr -y
!pip install -q pytesseract inference-sdk opencv-python pandas matplotlib tqdm

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


In [3]:
import os
import re
import cv2
import pandas as pd
import pytesseract
import matplotlib.pyplot as plt
from tqdm import tqdm
from inference_sdk import InferenceHTTPClient

In [5]:
!unzip -q /content/KSA_Dataset.zip -d /content/KSA_Dataset

csv_path = "/content/metadata_ground_truth_clean.csv"
df = pd.read_csv(csv_path)

print(df.shape)
df.head()

(414, 4)


,Original_Image,Distorted_Image_Path,Distortion_Type,Ground_Truth
0,car_223.jpg,KSA_License_Plate_Project/Gaussian_Blur/car_22...,Gaussian_Blur,6983 LNJ
1,car_223.jpg,KSA_License_Plate_Project/Motion_Blur/car_223.jpg,Motion_Blur,6983 LNJ
2,car_223.jpg,KSA_License_Plate_Project/Dirty/car_223.jpg,Dirty,6983 LNJ
3,car_223.jpg,KSA_License_Plate_Project/Occlusion/car_223.jpg,Occlusion,6983 LNJ
4,car_223.jpg,KSA_License_Plate_Project/Rotation/car_223.jpg,Rotation,6983 LNJ


In [ ]:
CLIENT = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key=""
)

MODEL_ID = "license-plate-recognition-rxg4e/11"

In [ ]:
DATASET_ROOT = "/content/KSA_Dataset/content"

def fix_path(path):
    return os.path.join(DATASET_ROOT, path)

def clean_text(text):
    text = text.upper()
    text = re.sub(r'[^A-Z0-9]', '', text)
    return text

def preprocess_plate(plate_crop):
    plate_big = cv2.resize(plate_crop, None, fx=5, fy=5, interpolation=cv2.INTER_CUBIC)
    return plate_big

def run_tesseract(plate_img):
    text = pytesseract.image_to_string(
        plate_img,
        config='--psm 6 -c tessedit_char_white'
    )
    return clean_text(text)

def detect_and_crop_plate(img_path):
    result = CLIENT.infer(img_path, model_id=MODEL_ID)

    if len(result["predictions"]) == 0:
        return None, result

    pred = result["predictions"][0]

    x = pred["x"]
    y = pred["y"]
    w = pred["width"]
    h = pred["height"]

    x1 = int(x - w / 2)
    y1 = int(y - h / 2)
    x2 = int(x + w / 2)
    y2 = int(y + h / 2)

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(img.shape[1], x2)
    y2 = min(img.shape[0], y2)

    plate_crop = img[y1:y2, x1:x2]

    return plate_crop, result

In [ ]:

from difflib import SequenceMatcher

def char_accuracy(gt, pred):
    return SequenceMatcher(None, gt, pred).ratio() * 100

distorted_results = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    img_path = fix_path(row["Distorted_Image_Path"])
    original_image = row["Original_Image"]
    distortion_type = row["Distortion_Type"]
    ground_truth = clean_text(row["Ground_Truth"])

    if not os.path.exists(img_path):
        distorted_results.append({
            "image_path": img_path,
            "original_image": original_image,
            "distortion_type": distortion_type,
            "ground_truth": ground_truth,
            "ocr_output": "",
            "image_accuracy": 0,
            "detection_confidence": 0,
            "correct": False,
            "status": "image_not_found"
        })
        continue

    try:
        # Detection
        plate_crop, detection_result = detect_and_crop_plate(img_path)

        if plate_crop is None:
            distorted_results.append({
                "image_path": img_path,
                "original_image": original_image,
                "distortion_type": distortion_type,
                "ground_truth": ground_truth,
                "ocr_output": "",
                "image_accuracy": 0,
                "detection_confidence": 0,
                "correct": False,
                "status": "no_plate_detected"
            })
            continue

        # Detection confidence
        prediction = detection_result["predictions"][0]
        detection_confidence = prediction["confidence"] * 100

        # Preprocess
        plate_big = cv2.resize(
            plate_crop,
            None,
            fx=5,
            fy=5,
            interpolation=cv2.INTER_CUBIC
        )

        # OCR
        text = pytesseract.image_to_string(
            plate_big,
            config='--psm 6 -c tessedit_char_whitelist=ABCDE'
        )

        ocr_output = clean_text(text)

        # Metrics
        image_acc = char_accuracy(ground_truth, ocr_output)
        correct = (ocr_output == ground_truth)

        distorted_results.append({
            "image_path": img_path,
            "original_image": original_image,
            "distortion_type": distortion_type,
            "ground_truth": ground_truth,
            "ocr_output": ocr_output,
            "image_accuracy": image_acc,
            "detection_confidence": detection_confidence,
            "correct": correct,
            "status": "success"
        })

    except Exception as e:
        distorted_results.append({
            "image_path": img_path,
            "original_image": original_image,
            "distortion_type": distortion_type,
            "ground_truth": ground_truth,
            "ocr_output": "",
            "image_accuracy": 0,
            "detection_confidence": 0,
            "correct": False,
            "status": str(e)
        })

distorted_results_df = pd.DataFrame(distorted_results)

overall_char_accuracy = distorted_results_df["image_accuracy"].mean()
overall_exact_match = distorted_results_df["correct"].mean() * 100
overall_detection_confidence = distorted_results_df["detection_confidence"].mean()

print(f"Overall Character Accuracy: {overall_char_accuracy:.2f}%")
print(f"Overall Exact Match Accuracy: {overall_exact_match:.2f}%")
print(f"Overall Detection Confidence: {overall_detection_confidence:.2f}%")

distorted_results_df.head()

100%|██████████| 414/414 [05:23<00:00,  1.28it/s]

Overall Character Accuracy: 28.54%
Overall Exact Match Accuracy: 2.17%
Overall Detection Confidence: 80.52%


,image_path,original_image,distortion_type,ground_truth,ocr_output,image_accuracy,detection_confidence,correct,status
0,/content/KSA_Dataset/content/KSA_License_Plate...,car_223.jpg,Gaussian_Blur,6983LNJ,S,0.000000,85.291541,False,success
1,/content/KSA_Dataset/content/KSA_License_Plate...,car_223.jpg,Motion_Blur,6983LNJ,,0.000000,81.386948,False,success
2,/content/KSA_Dataset/content/KSA_License_Plate...,car_223.jpg,Dirty,6983LNJ,,0.000000,87.548953,False,success
3,/content/KSA_Dataset/content/KSA_License_Plate...,car_223.jpg,Occlusion,6983LNJ,,0.000000,87.952799,False,success
4,/content/KSA_Dataset/content/KSA_License_Plate...,car_223.jpg,Rotation,6983LNJ,M3LNJO,61.538462,86.049962,False,success


In [9]:
distorted_summary = distorted_results_df.groupby("distortion_type").agg(
    total_images=("correct", "count"),
    correct_predictions=("correct", "sum"),
    exact_match_accuracy=("correct", lambda x: x.mean() * 100),
    avg_character_accuracy=("image_accuracy", "mean"),
    avg_detection_confidence=("detection_confidence", "mean")
).reset_index()

distorted_summary

,distortion_type,total_images,correct_predictions,exact_match_accuracy,avg_character_accuracy,avg_detection_confidence
0,Dirty,46,2,4.347826,56.721771,85.367690
1,Gaussian_Blur,46,0,0.000000,5.374396,74.519711
2,Low_Light,46,4,8.695652,54.799391,85.234130
3,Low_Res,46,0,0.000000,3.214883,74.962032
4,Motion_Blur,46,0,0.000000,2.871102,68.474411
5,Occlusion,46,2,4.347826,54.249233,84.918596
6,Over_Exposed,46,1,2.173913,44.797776,84.268034
7,Rotation,46,0,0.000000,19.358451,83.601117
8,Tilted,46,0,0.000000,15.446114,83.304637


In [ ]:
distorted_results_df.to_csv("/content/cv_ocr_distorted_results.csv", index=False)
distorted_summary.to_csv("/content/cv_ocr_distorted_summary.csv", index=False)

print("Saved distorted results and summary.")

In [ ]:

from difflib import SequenceMatcher

def char_accuracy(gt, pred):
    return SequenceMatcher(None, gt, pred).ratio() * 100

gt_map = (
    df[["Original_Image", "Ground_Truth"]]
    .drop_duplicates()
    .copy()
)

print("Number of labeled original images:", len(gt_map))

normal_results = []

for _, row in tqdm(gt_map.iterrows(), total=len(gt_map)):

    original_image = row["Original_Image"]
    ground_truth = clean_text(row["Ground_Truth"])

    img_path = f"/content/KSA_Dataset/content/KSA_License_Plate_Project/Normal/{original_image}"

    if not os.path.exists(img_path):
        normal_results.append({
            "image_path": img_path,
            "original_image": original_image,
            "distortion_type": "Normal",
            "ground_truth": ground_truth,
            "ocr_output": "",
            "image_accuracy": 0,
            "detection_confidence": 0,
            "correct": False,
            "status": "image_not_found"
        })
        continue

    try:
        plate_crop, detection_result = detect_and_crop_plate(img_path)

        if plate_crop is None:
            normal_results.append({
                "image_path": img_path,
                "original_image": original_image,
                "distortion_type": "Normal",
                "ground_truth": ground_truth,
                "ocr_output": "",
                "image_accuracy": 0,
                "detection_confidence": 0,
                "correct": False,
                "status": "no_plate_detected"
            })
            continue

        prediction = detection_result["predictions"][0]
        detection_confidence = prediction["confidence"] * 100

        plate_big = cv2.resize(
            plate_crop,
            None,
            fx=5,
            fy=5,
            interpolation=cv2.INTER_CUBIC
        )

        text = pytesseract.image_to_string(
            plate_big,
            config='--psm 6 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWX6789'
        )

        # تنظيف النص الناتج
        ocr_output = clean_text(text)

        # حساب الدقة لكل صورة
        image_acc = char_accuracy(ground_truth, ocr_output)

        # المطابقة الكاملة
        correct = (ocr_output == ground_truth)

        normal_results.append({
            "image_path": img_path,
            "original_image": original_image,
            "distortion_type": "Normal",
            "ground_truth": ground_truth,
            "ocr_output": ocr_output,
            "image_accuracy": image_acc,
            "detection_confidence": detection_confidence,
            "correct": correct,
            "status": "success"
        })

    except Exception as e:
        normal_results.append({
            "image_path": img_path,
            "original_image": original_image,
            "distortion_type": "Normal",
            "ground_truth": ground_truth,
            "ocr_output": "",
            "image_accuracy": 0,
            "detection_confidence": 0,
            "correct": False,
            "status": str(e)
        })

# تحويل النتائج لداتا فريم
normal_results_df = pd.DataFrame(normal_results)

# متوسط دقة الحروف
normal_avg_accuracy = normal_results_df["image_accuracy"].mean()

# دقة المطابقة الكاملة
normal_exact_match_accuracy = normal_results_df["correct"].mean() * 100

# متوسط ثقة الكشف
normal_avg_confidence = normal_results_df["detection_confidence"].mean()

print(f"\nNormal Images Average Character Accuracy: {normal_avg_accuracy:.2f}%")
print(f"Normal Images Exact Match Accuracy: {normal_exact_match_accuracy:.2f}%")
print(f"Normal Images Average Detection Confidence: {normal_avg_confidence:.2f}%")

normal_results_df.head()

Number of labeled original images: 46


100%|██████████| 46/46 [00:45<00:00,  1.01it/s]


Normal Images Average Character Accuracy: 57.22%
Normal Images Exact Match Accuracy: 8.70%
Normal Images Average Detection Confidence: 85.32%


,image_path,original_image,distortion_type,ground_truth,ocr_output,image_accuracy,detection_confidence,correct,status
0,/content/KSA_Dataset/content/KSA_License_Plate...,car_223.jpg,Normal,6983LNJ,L6983LNJ,93.333333,88.024443,False,success
1,/content/KSA_Dataset/content/KSA_License_Plate...,car_228.jpg,Normal,999GLG,3994CL,50.000000,81.434131,False,success
2,/content/KSA_Dataset/content/KSA_License_Plate...,car_198.jpg,Normal,6881ULD,VAAY3536881ULDS,63.636364,86.084181,False,success
3,/content/KSA_Dataset/content/KSA_License_Plate...,car_193.jpg,Normal,3297UKB,3297UKB,100.000000,86.067820,True,success
4,/content/KSA_Dataset/content/KSA_License_Plate...,car_204.jpg,Normal,2480AJD,PCSAY32800,35.294118,84.162420,False,success


In [ ]:
normal_results_df.to_csv("/content/cv_ocr_normal_results.csv", index=False)
print("Saved: /content/cv_ocr_normal_results.csv")